# Gold Customer Activity Report

This notebook builds two Iceberg gold-layer tables from the Bronze CDC event log:

- `lakehouse.cdc.gold_customer_activity`
- `lakehouse.cdc.gold_customer_churn`

It tracks each customer's lifecycle, profile changes, current status, and potential churn/staleness.


## 1. Spark package setup

Run this cell before creating the Spark session. It makes sure the notebook kernel starts PySpark with the same packages used by `spark-submit`.


In [3]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0,"
    "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
    "org.apache.iceberg:iceberg-aws-bundle:1.10.0 "
    "pyspark-shell"
)

print("✅ PYSPARK_SUBMIT_ARGS configured")

✅ PYSPARK_SUBMIT_ARGS configured


## 2. Create Spark session


In [44]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

print("🚀 Starting Gold Customer Activity notebook...")

spark = (
    SparkSession.builder
    .appName("CDC-Gold-CustomerActivity")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .getOrCreate()
)

print("✅ Spark session created")

🚀 Starting Gold Customer Activity notebook...
✅ Spark session created


## 3. Read Bronze and Silver source tables


In [45]:
bronze_df = spark.sql("SELECT * FROM lakehouse.cdc.bronze_cdc")
silver_customers = spark.sql("SELECT id FROM lakehouse.cdc.silver_customers")

customers_bronze = (
    bronze_df
    .filter(F.col("topic") == "dbserver1.public.customers")
    .withColumn(
        "entity_id",
        F.coalesce(
            F.get_json_object("after", "$.id"),
            F.get_json_object("before", "$.id")
        ).cast("int")
    )
    .filter(F.col("entity_id").isNotNull())
    .withColumn("event_ts", (F.col("ts_ms") / 1000).cast("timestamp"))
)

print(f"📥 Customer CDC events (including deletes): {customers_bronze.count()}")
customers_bronze.select("topic", "entity_id", "op", "event_ts", "kafka_offset").show(10, truncate=False)

📥 Customer CDC events (including deletes): 1292
+--------------------------+---------+---+-----------------------+------------+
|topic                     |entity_id|op |event_ts               |kafka_offset|
+--------------------------+---------+---+-----------------------+------------+
|dbserver1.public.customers|840      |r  |2026-04-29 07:16:44.119|0           |
|dbserver1.public.customers|672      |r  |2026-04-29 07:16:44.135|1           |
|dbserver1.public.customers|416      |r  |2026-04-29 07:16:44.136|2           |
|dbserver1.public.customers|1932     |r  |2026-04-29 07:16:44.138|3           |
|dbserver1.public.customers|1706     |r  |2026-04-29 07:16:44.139|4           |
|dbserver1.public.customers|260      |r  |2026-04-29 07:16:44.139|5           |
|dbserver1.public.customers|1945     |r  |2026-04-29 07:16:44.139|6           |
|dbserver1.public.customers|804      |r  |2026-04-29 07:16:44.139|7           |
|dbserver1.public.customers|1483     |r  |2026-04-29 07:16:44.14 |8     

In [46]:
spark.sql("""
SELECT
    op,
    before,
    after,
    ts_ms
FROM lakehouse.cdc.bronze_cdc
WHERE topic = 'dbserver1.public.customers'
LIMIT 30
""").show(truncate=False)

+---+------+----------------------------------------------------------------------------------------------------------------------------+-------------+
|op |before|after                                                                                                                       |ts_ms        |
+---+------+----------------------------------------------------------------------------------------------------------------------------+-------------+
|r  |NULL  |{"id":840,"name":"Diego Kowalski","email":"updated_840_552@inbox.org","country":"Estonia","created_at":1777400882936444}    |1777447004119|
|r  |NULL  |{"id":672,"name":"Noah Park","email":"noah.park@inbox.org","country":"Japan","created_at":1777400143816223}                 |1777447004135|
|r  |NULL  |{"id":416,"name":"Yuki Smith","email":"yuki.smith@test.net","country":"India","created_at":1777399044271992}                |1777447004136|
|r  |NULL  |{"id":1932,"name":"Emma Park","email":"emma.park@mail.com","country":"Norway

## 4. Detect email and country changes


In [47]:
w_asc = Window.partitionBy("entity_id").orderBy(F.col("ts_ms").asc())

customers_with_prev = (
    customers_bronze
    .withColumn("prev_after", F.lag("after").over(w_asc))
    .withColumn("prev_email", F.get_json_object("prev_after", "$.email"))
    .withColumn("prev_country", F.get_json_object("prev_after", "$.country"))
    .withColumn("cur_email", F.get_json_object("after", "$.email"))
    .withColumn("cur_country", F.get_json_object("after", "$.country"))
    .withColumn(
        "email_changed",
        F.when(
            F.col("prev_after").isNotNull()
            & F.col("cur_email").isNotNull()
            & F.col("prev_email").isNotNull()
            & (F.col("cur_email") != F.col("prev_email")),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "country_changed",
        F.when(
            F.col("prev_after").isNotNull()
            & F.col("cur_country").isNotNull()
            & F.col("prev_country").isNotNull()
            & (F.col("cur_country") != F.col("prev_country")),
            1,
        ).otherwise(0),
    )
)

customers_with_prev.select(
    "entity_id", "op", "event_ts", "cur_email", "prev_email", "email_changed",
    "cur_country", "prev_country", "country_changed"
).show(10, truncate=False)

+---------+---+-----------------------+-----------------+-----------------+-------------+-----------+------------+---------------+
|entity_id|op |event_ts               |cur_email        |prev_email       |email_changed|cur_country|prev_country|country_changed|
+---------+---+-----------------------+-----------------+-----------------+-------------+-----------+------------+---------------+
|1        |c  |2026-04-29 07:36:07.893|alice@example.com|NULL             |0            |Estonia    |NULL        |0              |
|1        |d  |2026-04-29 07:39:19.823|NULL             |alice@example.com|0            |NULL       |Estonia     |0              |
|2        |c  |2026-04-29 07:36:07.894|bob@example.com  |NULL             |0            |Finland    |NULL        |0              |
|2        |d  |2026-04-29 07:39:26.917|NULL             |bob@example.com  |0            |NULL       |Finland     |0              |
|3        |c  |2026-04-29 07:36:07.894|carol@example.com|NULL             |0       

## 5. Build `gold_customer_activity` DataFrame


In [48]:
agg = customers_with_prev.groupBy("entity_id").agg(
    F.min(
        (F.get_json_object(F.col("after"), "$.created_at") / 1_000_000).cast("timestamp")
    ).alias("first_seen_ts"),
    F.count("*").alias("total_events"),
    F.sum("email_changed").alias("email_changes"),
    F.sum("country_changed").alias("country_changes"),
    F.max("event_ts").alias("last_change_ts"),
    F.max(F.when(F.col("op") == "d", 1).otherwise(0)).alias("ever_deleted"),
    F.max(F.when(F.col("op") == "d", F.col("event_ts"))).alias("deleted_at_ts"),
)

silver_ids = silver_customers.select(F.col("id").alias("silver_id"))

gold_df = (
    agg
    .join(silver_ids, agg["entity_id"] == silver_ids["silver_id"], "left")
    .withColumn(
        "current_status",
        F.when(
            F.col("ever_deleted") == 1, "deleted"
        ).when(
            F.col("silver_id").isNotNull(), "active"
        ).otherwise("deleted"),
    )
    .withColumn(
        "days_since_last_change",
        F.datediff(F.current_timestamp().cast("date"), F.col("last_change_ts").cast("date")),
    )
    .withColumn(
        "fields_changed_most_often",
        F.when(F.col("email_changes") > F.col("country_changes"), "email")
        .when(F.col("country_changes") > F.col("email_changes"), "country")
        .when((F.col("email_changes") == F.col("country_changes")) & (F.col("email_changes") > 0), "email,country")
        .otherwise("none"),
    )
    .select(
        F.col("entity_id").alias("customer_id"),
        F.col("first_seen_ts"),
        F.col("total_events"),
        F.col("email_changes"),
        F.col("country_changes"),
        F.col("fields_changed_most_often"),
        F.col("current_status"),
        F.col("days_since_last_change"),
        F.col("last_change_ts"),
        F.col("ever_deleted").cast("boolean").alias("ever_deleted"),
        F.col("deleted_at_ts"),
    )
)

print("✅ Gold aggregations computed")
gold_df.show(20, truncate=False)

✅ Gold aggregations computed
+-----------+--------------------------+------------+-------------+---------------+-------------------------+--------------+----------------------+-----------------------+------------+-----------------------+
|customer_id|first_seen_ts             |total_events|email_changes|country_changes|fields_changed_most_often|current_status|days_since_last_change|last_change_ts         |ever_deleted|deleted_at_ts          |
+-----------+--------------------------+------------+-------------+---------------+-------------------------+--------------+----------------------+-----------------------+------------+-----------------------+
|1          |2026-04-29 07:36:07.47827 |2           |0            |0              |none                     |deleted       |0                     |2026-04-29 07:39:19.823|true        |2026-04-29 07:39:19.823|
|2          |2026-04-29 07:36:07.47827 |2           |0            |0              |none                     |deleted       |0          

## 6. Write `gold_customer_activity` Iceberg table


In [49]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.cdc.gold_customer_activity (
        customer_id                   INT,
        first_seen_ts                 TIMESTAMP,
        total_events                  BIGINT,
        email_changes                 BIGINT,
        country_changes               BIGINT,
        fields_changed_most_often     STRING,
        current_status                STRING,
        days_since_last_change        INT,
        last_change_ts                TIMESTAMP,
        ever_deleted                  BOOLEAN,
        deleted_at_ts                 TIMESTAMP
    ) USING iceberg
""")

gold_df.writeTo("lakehouse.cdc.gold_customer_activity").createOrReplace()

print("✅ gold_customer_activity written")

✅ gold_customer_activity written


## 7. Build `gold_customer_churn`

The assignment asks for a view, but this Iceberg catalog does not support views, so this notebook creates an Iceberg table with the same result.


In [50]:
spark.sql("""
    CREATE OR REPLACE TABLE lakehouse.cdc.gold_customer_churn AS
    SELECT
        customer_id,
        first_seen_ts,
        deleted_at_ts,
        last_change_ts,
        total_events,
        days_since_last_change,
        CASE
            WHEN current_status = 'deleted'
                 AND deleted_at_ts >= (current_timestamp() - INTERVAL 24 HOURS)
                 OR (current_status = 'active'AND days_since_last_change >= 7)
            THEN 'deleted_last_24h'
            WHEN days_since_last_change >= 7
            THEN 'stale_7_days'
        END AS churn_flag
    FROM lakehouse.cdc.gold_customer_activity
    WHERE
        (current_status = 'deleted'
         AND deleted_at_ts >= (current_timestamp() - INTERVAL 24 HOURS))
        OR days_since_last_change >= 7
""")

print("✅ gold_customer_churn table created")

✅ gold_customer_churn table created


## 8. Preview results


In [42]:
print("📊 gold_customer_activity")
spark.sql("""
    SELECT
        customer_id,
        current_status,
        total_events,
        email_changes,
        country_changes,
        fields_changed_most_often,
        days_since_last_change,
        date(first_seen_ts) AS first_seen,
        date(last_change_ts) AS last_changed
    FROM lakehouse.cdc.gold_customer_activity
    ORDER BY customer_id
""").show(20, truncate=False)

print("📊 gold_customer_churn")
spark.sql("""
    SELECT *
    FROM lakehouse.cdc.gold_customer_churn
    ORDER BY churn_flag, customer_id
""").show(20, truncate=False)

📊 gold_customer_activity
+-----------+--------------+------------+-------------+---------------+-------------------------+----------------------+----------+------------+
|customer_id|current_status|total_events|email_changes|country_changes|fields_changed_most_often|days_since_last_change|first_seen|last_changed|
+-----------+--------------+------------+-------------+---------------+-------------------------+----------------------+----------+------------+
|1          |deleted       |2           |0            |0              |none                     |0                     |2026-04-29|2026-04-29  |
|2          |deleted       |2           |0            |0              |none                     |0                     |2026-04-29|2026-04-29  |
|3          |deleted       |3           |0            |1              |country                  |0                     |2026-04-29|2026-04-29  |
|4          |active        |10          |5            |4              |email                    |0       

## 9. Analysis queries


In [51]:
print("🔍 Q1 — Deleted customers and active duration")
spark.sql("""
    SELECT
        customer_id,
        date(first_seen_ts) AS first_seen,
        date(deleted_at_ts) AS deleted_at,
        datediff(deleted_at_ts, first_seen_ts) AS days_active,
        total_events
    FROM lakehouse.cdc.gold_customer_activity
    WHERE current_status = 'deleted'
    ORDER BY days_active DESC
""").show(20, truncate=False)

print("🔍 Q2 — Customers with most profile changes")
spark.sql("""
    SELECT
        customer_id,
        total_events,
        email_changes,
        country_changes,
        fields_changed_most_often,
        current_status
    FROM lakehouse.cdc.gold_customer_activity
    ORDER BY total_events DESC
    LIMIT 5
""").show(truncate=False)

🔍 Q1 — Deleted customers and active duration
+-----------+----------+----------+-----------+------------+
|customer_id|first_seen|deleted_at|days_active|total_events|
+-----------+----------+----------+-----------+------------+
|36         |2026-04-28|2026-04-29|1          |4           |
|1          |2026-04-29|2026-04-29|0          |2           |
|2          |2026-04-29|2026-04-29|0          |2           |
|3          |2026-04-29|2026-04-29|0          |3           |
|5          |2026-04-29|2026-04-29|0          |3           |
|6          |2026-04-29|2026-04-29|0          |2           |
|7          |2026-04-29|2026-04-29|0          |2           |
|8          |2026-04-29|2026-04-29|0          |6           |
|9          |2026-04-29|2026-04-29|0          |5           |
|10         |2026-04-29|2026-04-29|0          |2           |
|11         |2026-04-29|2026-04-29|0          |5           |
|12         |2026-04-29|2026-04-29|0          |3           |
|13         |2026-04-29|2026-04-29|0    

## 10. Finish


In [52]:
spark.sql("""
    SELECT
        current_status,
        count(*) AS n,
        min(days_since_last_change) AS min_days,
        max(days_since_last_change) AS max_days,
        sum(CASE WHEN ever_deleted THEN 1 ELSE 0 END) AS deleted_count
    FROM lakehouse.cdc.gold_customer_activity
    GROUP BY current_status
""").show()

+--------------+----+--------+--------+-------------+
|current_status|   n|min_days|max_days|deleted_count|
+--------------+----+--------+--------+-------------+
|        active|1108|       0|       0|            0|
|       deleted|  48|       0|       0|           48|
+--------------+----+--------+--------+-------------+



In [37]:
spark.sql("""
SELECT
    op,
    coalesce(
        get_json_object(after, '$.id'),
        get_json_object(before, '$.id')
    ) AS customer_id,
    get_json_object(before, '$.email') AS before_email,
    get_json_object(after, '$.email') AS after_email,
    get_json_object(before, '$.country') AS before_country,
    get_json_object(after, '$.country') AS after_country,
    ts_ms,
    kafka_offset
FROM lakehouse.cdc.bronze_cdc
WHERE topic = 'dbserver1.public.customers'
  AND coalesce(
      get_json_object(after, '$.id'),
      get_json_object(before, '$.id')
  ) = '1'
ORDER BY ts_ms, kafka_offset
""").show(truncate=False)

+---+-----------+------------+-----------------+--------------+-------------+-------------+------------+
|op |customer_id|before_email|after_email      |before_country|after_country|ts_ms        |kafka_offset|
+---+-----------+------------+-----------------+--------------+-------------+-------------+------------+
|c  |1          |NULL        |alice@example.com|NULL          |Estonia      |1777448167893|1051        |
|d  |1          |            |NULL             |              |NULL         |1777448359823|1063        |
+---+-----------+------------+-----------------+--------------+-------------+-------------+------------+



In [53]:
spark.sql("""
    SELECT
        op,
        count(*) AS n
    FROM lakehouse.cdc.bronze_cdc
    WHERE topic = 'dbserver1.public.customers'
    GROUP BY op
    ORDER BY op
""").show()

+---+----+
| op|   n|
+---+----+
|  c| 108|
|  d|  48|
|  r|1051|
|  u|  85|
+---+----+

